In [ ]:
# ============================================================
# CONFIGURATION - every path comes from config/paths.py, the single
# source of truth. Override cluster locations with the MUSICA_ENV_*
# environment variables documented there. Do not hard-code paths here.
# ============================================================
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
_ROOT = next(p for p in [_here, *_here.parents]
             if (p / 'config' / 'paths.py').exists())
sys.path.insert(0, str(_ROOT))
import config  # also puts functions/ on sys.path
from config import paths as P


This script works on addressing need to get estimates of % O3 produced versus imported

In [ ]:
import os
import glob
import fnmatch

import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point, Polygon

import xarray as xr
import numpy as np

import matplotlib.pyplot as plt # Core library for plotting
import matplotlib.cm as cm # To use different colormaps
import cartopy.crs as ccrs # For map projection
import seaborn as sns # boxplot


In [ ]:
# !pip install geopandas

In [ ]:
import sys
sys.path.insert(0,f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/')

from Plot_2D import Plot_2D # To draw a map

SCRIP_CONUS = f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/ne0CONUS_ne30x8_np4_SCRIP.nc'
SCRIP_ne30 = f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/ne30np4_091226_pentagons.nc'

Download the shape files   
Website: https://www.census.gov/geographies/mapping-files/time-series/geo/carto-boundary-file.html
 
Copied to Svante: rsync -avz --exclude=".*" -e ssh /Users/$USER/Downloads/USCensus/* "<username>@svante9.mit.edu:{P.HOME_ROOT}/HelpfulFiles/USCensus2018/"

Unzip use unzip_to_folder.sh in the Helpful US Census 2018 folder with chmod +x unzip_to_folder.sh

./unzip_to_folder.sh yourfile.zip

In [ ]:
def molecules_to_kg_per_m2_per_s(molecules_per_cm2_per_s, molecular_weight):
    """
        Read in emissions data in [molecules cm-2 s-1], and molecular_weight in [g/mole]
        Return emissions in [kg m-2 s-1]    
    """
    # Constants
    avogadro_number = 6.022e23  # molecules per mole
    m2_to_cm2 = 1e4  # square meters to square centimeters 
    kg_to_g = 1e3

    # Convert 
    kg_per_m2_per_s = molecules_per_cm2_per_s*(1/avogadro_number)*molecular_weight*(1/kg_to_g)*(m2_to_cm2)

    return kg_per_m2_per_s

# # Example usage
# molecules_per_cm2_per_s = 1  # for example, 1e18 molecules per cm^2 per second
# molecular_weight = 30  # molecular weight of water in g/mol

# result = molecules_to_kg_per_m2_per_s(molecules_per_cm2_per_s, molecular_weight)
# print("Result:", result, "kg/m^2/s")


setunit = r'$kg$ $m^{-2}s^{-1}$'

# Set up the shape file

In [ ]:
### Geo information for boundaries
from shapely.geometry import Point
import numpy as np
import xarray as xr
import geopandas as gpd
import geopandas as gpd

USCensus_diri = f'{P.HOME_ROOT}/HelpfulFiles/USCensus2018/'

# # Path to the shapefile (adjust if the file is inside a folder)
# shapefile_path = '{P.HOME_ROOT}/HelpfulFiles/world-administrative-boundaries/world-administrative-boundaries.shp'

# # Read shapefile directly
# gdf = gpd.read_file(shapefile_path)

# # # Preview
# # print(gdf.columns)
# # print(gdf.head())

# # Adjust based on the actual column names (use print(gdf.columns) to confirm)
# us_boundary = gdf[(gdf['status'] == 'Member State')&(gdf['name'] == 'United States of America')]  # or 'USA', depending on the file

# # Create a GeoDataFrame
# us_gdf = gpd.GeoDataFrame(us_boundary, geometry='geometry')



In [ ]:
shapefile_path = f'{USCensus_diri}cb_2018_us_state_500k/cb_2018_us_state_500k.shp'
print(shapefile_path)

gdf = gpd.read_file(shapefile_path)
# Print basic information
print(gdf.info())           # Column names, data types, non-null counts
# print(gdf.head())           # Preview the first few rows
# print(gdf.columns)          # List all column names
# print(gdf.crs)              # Coordinate reference system (CRS)
# print(gdf.geometry.name)    # Name of the geometry column
# print(gdf.geometry.type.unique())  # Geometry types (e.g., Polygon, MultiPolygon)

# Optional: plot a quick map
# gdf.plot()


In [ ]:
# gdf = gpd.read_file(shapefile_path)
# gdf

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Load the shapefile
shapefile_path = f"{P.HOME_ROOT}/HelpfulFiles/USCensus2018/cb_2018_us_state_500k/cb_2018_us_state_500k.shp"
gdf = gpd.read_file(shapefile_path)

# Filter to lower 48 states
exclude = ['AK', 'HI', 'PR', 'GU', 'VI', 'MP', 'AS']
gdf_lower48 = gdf[~gdf['STUSPS'].isin(exclude)]

# Project to an equal-area CRS for buffering
gdf_proj = gdf_lower48.to_crs(epsg=5070)

# Merge and buffer
merged_shape = gdf_proj.unary_union
# buffered_shape = merged_shape.buffer(100000)  # 100 km
buffered_shape = merged_shape.buffer(80000)  # 80 km

# Create a GeoDataFrame for plotting
gdf_buffered = gpd.GeoDataFrame(geometry=[buffered_shape], crs=gdf_proj.crs)

# Reproject both back to geographic for consistent plotting
gdf_lower48 = gdf_lower48.to_crs(epsg=4326)
gdf_buffered = gdf_buffered.to_crs(epsg=4326)


### To make sure there is no hole
from shapely.geometry import Polygon, MultiPolygon

def remove_holes(geometry):
    if isinstance(geometry, Polygon):
        return Polygon(geometry.exterior)
    elif isinstance(geometry, MultiPolygon):
        return MultiPolygon([Polygon(p.exterior) for p in geometry.geoms])
    else:
        return geometry  # return as-is if not polygonal

# Apply hole removal to your geometry
geometry_no_holes = remove_holes(gdf_buffered.iloc[0].geometry)

# Update gdf_buffered
gdf_buffered['geometry'] = [geometry_no_holes]


In [ ]:
gdf_buffered

In [ ]:
gdf_buffered

In [ ]:
import matplotlib.patches as mpatches

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
gdf_lower48.plot(ax=ax, color='lightblue', edgecolor='black')
gdf_buffered.plot(ax=ax, color='none', edgecolor='red', linewidth=2)

# Create custom legend handles
original_patch = mpatches.Patch(color='lightblue', label='Original Lower 48 States')
buffer_patch = mpatches.Patch(edgecolor='red', facecolor='none', linewidth=2, label='Buffered 80 km')

plt.legend(handles=[original_patch, buffer_patch], loc='lower left',fontsize=20,)
plt.title("Lower 48 U.S. States with 80 km Coastal Buffer", fontsize=30, y=1.02)
plt.grid(True)

# Enlarge tick label font size
ax.tick_params(axis='both', which='major', labelsize=20)

# Tighten the layout
plt.tight_layout()

# Save the figure
# plt.savefig(savefig_filename, dpi=400, bbox_inches='tight')  # Adjust the filename and dpi as needed

plt.show()


In [ ]:
import matplotlib.patches as mpatches

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
gdf_lower48.plot(ax=ax, color='lightblue', edgecolor='black')
gdf_buffered.plot(ax=ax, color='none', edgecolor='red', linewidth=2)

# Create custom legend handles
original_patch = mpatches.Patch(color='lightblue', label='Original Lower 48 States')
# buffer_patch = mpatches.Patch(edgecolor='red', facecolor='none', linewidth=2, label='Buffered 100 km')
buffer_patch = mpatches.Patch(edgecolor='red', facecolor='none', linewidth=2, label='Buffered 50 km')

plt.legend(handles=[original_patch, buffer_patch], loc='lower left',fontsize=20,)
# plt.title("Lower 48 U.S. States with 100 km Coastal Buffer", fontsize=30, y=1.02)
plt.title("Lower 48 U.S. States with 50 km Coastal Buffer", fontsize=30, y=1.02)
plt.grid(True)

# Enlarge tick label font size
ax.tick_params(axis='both', which='major', labelsize=20)

# Tighten the layout
plt.tight_layout()

# Save the figure
# plt.savefig(savefig_filename, dpi=400, bbox_inches='tight')  # Adjust the filename and dpi as needed

plt.show()


# Remove ANT emissions over CONUS Box-land

In [ ]:
### Read in original emissions data
CAMS_diri = f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4/'

CONUSlandMasked_diri = f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4_CONUSlandMasked_c20250619/'

In [ ]:
# Search for files that match the pattern

import glob
import os

# Directory where files are stored
# CAMS_diri = '/path/to/your/files'  # Make sure this is correctly defined

# Updated pattern with wildcard
pattern = os.path.join(CAMS_diri, 'CAMS-GLOB-ANT_ne30np4_*_v6.2_monthly.nc')

# Get list of matching files in full path
CAMS_file_list = glob.glob(pattern)
# print(CAMS_v51_file_list)

# Extract only the filenames
CAMS_file_names = [os.path.basename(f) for f in CAMS_file_list]

print(CAMS_file_names[:3])


In [ ]:
spc_ls = []
for spcIdx in range(len(CAMS_file_names)):
    spc_name = CAMS_file_names[spcIdx].split('_')[4]
    spc_ls.append(spc_name)

In [ ]:
import re
# Extract species name between 'ne30np4_' and '_c20210423'
species_names = [
    re.search(r'ne30np4_(.+?)_v6.2_monthly', f).group(1)
    for f in CAMS_file_names
]
species_names = sorted(species_names)
print(species_names)

In [ ]:
len(species_names)

In [ ]:
# spc_fileOUTpath = f'{CAMS_diri}CAMS-GLOB-ANT_ne30np4_{spc}_v6.2_monthly.nc'

In [ ]:
Abs_rangeMax_dic = {'NO':7e-11,
                  'CO':5e-10,
                  'SO2':3e-11, 
                  'bc_a4':3e-12,
                  'NH3':2e-11,
                  'C2H2':2e-12, 
                  'C2H4':5e-12, # ethene
                  'C2H5OH':5e-11,
                  'C2H6':6e-12, # ethane
                  'C3H6':2e-11, 
                  'C3H8':2e-12, 
                  'CH3OH':2e-12,
                  'CH3CHO':2e-12, # 
                  'CH3COCH3':2e-12, 
                  'MEK':2e-12,
                    
                # Second panel
                  'BENZENE':3e-12, 
                  'TOLUENE':7e-12,
                  'BIGENE':4e-12,   
                  'MTERP':5e-13, 
                  'ISOP':8e-14, 
                  'CH2O':3e-12, # formaldehyde
                  'SVOC':6e-12,
                   }

## Note that the magnitude for some VOC species is much lower compared to their biogenic emissions
# MTERP: 4.90e-13 | 2.14e-13 (anthrop) compared to 
# ISOP: 8.37e-14 | 7.34e-14 (anthrop) compared to 

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'DejaVu Sans'  # Change to a font that supports the superscript characters

def get_superscript(magnitude):
    return r'$10^{' + str(magnitude) + '}$'

### Create Maskds andRemove for a list of species files

In [ ]:
Bufferopt = '80kmBuffer'

### Create the Masked array and save to nc file | use NO
spc = 'NO'
spc_fileINpath = f'{CAMS_diri}CAMS-GLOB-ANT_ne30np4_{spc}_v6.2_monthly.nc'
# Open Xarray dataset
spci_ds = xr.open_dataset(spc_fileINpath)

# MUSICA lons goes from 0-360, convert it to +-180 | Need to adjust lon_right and lon_left for MUSICA!!
lat = spci_ds['lat'].values
lon = spci_ds['lon'].values
Adjustedlons = np.where(lon >= 180, lon - 360, lon)
# Create a DataArray for the adjusted lon
Adjustedlons_da = xr.DataArray(
                                Adjustedlons,  
                                dims=('ncol'),
                                coords={'ncol': spci_ds.ncol.values} 
                                )

# Add the DataArray back to the dataset
spci_ds['Adjustedlons'] = Adjustedlons_da

### To process for all months for the given period (reduce the size of processed dataset)
Timei = '2022-07-31T00:00:00.000000000'
MonthiMean_ds = spci_ds.sel(time=Timei, method="nearest")
latitudes = MonthiMean_ds.lat.values
longitudes = MonthiMean_ds.Adjustedlons.values
# MonthiMean_ds

# Step 1: Define CONUS bounding box
lon_lefti = -66.95   # Eastern edge
lon_righti = -125.0  # Western edge
lat_boti = 24.4      # Southern edge
lat_upi = 49.5       # Northern edge

# Step 2: Create a combined mask
mask = np.zeros(len(lat), dtype=bool)

for geom in gdf_buffered['geometry']:
    for i in range(len(lat)):
        lon = Adjustedlons[i]
        lati = lat[i]
        point = Point(lon, lati)

        # Only mask if inside CONUS bounding box and also within the geometry
        in_box = (lon >= lon_righti) & (lon <= lon_lefti) & (lati >= lat_boti) & (lati <= lat_upi)
        if in_box and geom.contains(point):
            mask[i] = True

# Step 3: Convert to xarray mask
mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": spci_ds["ncol"]})

# Step 4: Apply the mask to remove CONUS land pixels
masked_ds = MonthiMean_ds.where(~mask_da, drop=False)
# sum_masked = masked_ds['sum']

### Can probably save this masked array to use later 
maskfilePath = f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/ne30np4_091226_pentagons_CONUSlandMaskedFalse_{Bufferopt}.nc'
# Save mask_da to a NetCDF file
mask_da.to_netcdf(maskfilePath)
print('Save to:', maskfilePath)

In [ ]:
Bufferopt = '80kmBuffer'
ne30maskfilePath = f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/ne30np4_091226_pentagons_CONUSlandMaskedFalse_{Bufferopt}.nc'
# Read the saved NetCDF file
mask_da_loaded = xr.open_dataarray(ne30maskfilePath)
mask_da_loaded

In [ ]:
# masked_ds = MonthiMean_ds.where(~mask_da_loaded, drop=False)
# masked_ds

In [ ]:
targetspc_ls = ['ISOP', 'NO']
timeslice_start = '2021-11-30T00:00:00.000000000'
timeslice_end = '2024-01-30T00:00:00.000000000'

for spc in targetspc_ls:
    spc_fileINpath = f'{CAMS_diri}CAMS-GLOB-ANT_ne30np4_{spc}_v6.2_monthly.nc'
    
    # Open Xarray dataset
    spci_ds = xr.open_dataset(spc_fileINpath)

    # MUSICA lons goes from 0-360, convert it to +-180 | Need to adjust lon_right and lon_left for MUSICA!!
    lat = spci_ds['lat'].values
    lon = spci_ds['lon'].values
    Adjustedlons = np.where(lon >= 180, lon - 360, lon)
    # Create a DataArray for the adjusted lon
    Adjustedlons_da = xr.DataArray(
                                    Adjustedlons,  
                                    dims=('ncol'),
                                    coords={'ncol': spci_ds.ncol.values} 
                                    )

    # Add the DataArray back to the dataset
    spci_ds['Adjustedlons'] = Adjustedlons_da
    
    ### Use the mask read in to remove pixels over CONUS-land, need to use zero instead of nan
    sum_masked = spci_ds['sum'].where(~mask_da_loaded, other=0)

    spci_ds['sum'] = sum_masked

    # Remove the 'Adjustedlons' variable from the dataset
    spci_ds = spci_ds.drop_vars('Adjustedlons')
    
    ### Save masked to the corresponding file
    spc_fileOUTpath = f'{CONUSlandMasked_diri}CAMS-GLOB-ANT_ne30np4_{spc}_v6.2_monthly.nc'
    spci_ds.to_netcdf(spc_fileOUTpath)
    print('Save to:',spc_fileOUTpath)

In [ ]:
masked_ds = spci_ds

In [ ]:
testfile = f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4_CONUSlandMasked_c20250619/CAMS-GLOB-ANT_ne30np4_NO_v6.2_monthly.nc'
test_ds = xr.open_dataset(testfile)

### Plotting
# get rangeMax
rangeMax = Abs_rangeMax_dic[spc]

### Accounting for scientific notation
# Get the magnitude in scientific notation, adjust for e-
magnitude = int(np.floor(np.log10(abs(rangeMax))))
# Format the magnitude as a superscript
formatted_magnitude = get_superscript(str(magnitude))
modified_rangeMax = rangeMax*(1/(10)**magnitude)

# Quick plot to check
Timei = '2022-07-01T00:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne30)"""
PlotRegion = 'CONUS'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = test_ds.sel(time=Timei,method='nearest')['sum'].values#*scalefactor

vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, spci_ds['sum'].molecular_weight)
Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
longname = test_ds['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15, twodecimal=True ) 
elif PlotRegion=="Global":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            state=True, 
              grid_line=False, grid_line_lw=0.15, twodecimal=True ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
testfile = f'{CAMS_diri}CAMS-GLOB-ANT_ne30np4_NO_v6.2_monthly.nc'
test_ds = xr.open_dataset(testfile)

### Plotting
# get rangeMax
rangeMax = Abs_rangeMax_dic[spc]

### Accounting for scientific notation
# Get the magnitude in scientific notation, adjust for e-
magnitude = int(np.floor(np.log10(abs(rangeMax))))
# Format the magnitude as a superscript
formatted_magnitude = get_superscript(str(magnitude))
modified_rangeMax = rangeMax*(1/(10)**magnitude)

# Quick plot to check
Timei = '2022-07-01T00:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne30)"""
PlotRegion = 'CONUS'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = test_ds.sel(time=Timei,method='nearest')['sum'].values#*scalefactor

vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, spci_ds['sum'].molecular_weight)
Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
longname = masked_ds['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15, twodecimal=True ) 
elif PlotRegion=="Global":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            state=True, 
              grid_line=False, grid_line_lw=0.15, twodecimal=True ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
CONUSlandMasked_diri

In [ ]:
## Read back
for spc in targetspc_ls:
    spc_maskedINpath = f'{CONUSlandMasked_diri}CAMS-GLOB-ANT_ne30np4_{spc}_v6.2_monthly.nc'
    masked_ds = xr.open_dataset(spc_maskedINpath)

    ### Plotting
    # get rangeMax
    rangeMax = Abs_rangeMax_dic[spc]

    ### Accounting for scientific notation
    # Get the magnitude in scientific notation, adjust for e-
    magnitude = int(np.floor(np.log10(abs(rangeMax))))
    # Format the magnitude as a superscript
    formatted_magnitude = get_superscript(str(magnitude))
    modified_rangeMax = rangeMax*(1/(10)**magnitude)
    
    # Quick plot to check
    Timei = '2022-07-01T00:00:00.000000000'

    setmap = 'viridis'

    """ds_mappedMUSICA (regridded to ne30)"""
    PlotRegion = 'CONUS'

    # scalefactor = 1e-11
    # formatted_label = f"{scalefactor:.0e}" 

    Plot_ar = masked_ds.sel(time=Timei,method='nearest')['sum'].values#*scalefactor

    vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, spci_ds['sum'].molecular_weight)
    Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
    longname = masked_ds['sum'].long_name
    rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

    ### Which map
    fig = plt.figure( figsize=(8,6) ) 
    # - ne30x8 regional refinement over CONUS|
    ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
    if PlotRegion=="CONUS":
        im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
                cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
                unit=f'{formatted_magnitude}{Plot_unit}',
                state=True, lon_range=[-140,-50], lat_range=[15,60],
                  grid_line=False, grid_line_lw=0.15, twodecimal=True ) 
    elif PlotRegion=="Global":
        im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
                cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
                unit=f'{formatted_magnitude}{Plot_unit}',
                state=True, 
                  grid_line=False, grid_line_lw=0.15, twodecimal=True ) 

    plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
masked_ds.time.values

In [ ]:
## Read back
for spc in targetspc_ls:
    spc_maskedINpath = f'{CONUSlandMasked_diri}CAMS-GLOB-ANT_ne30np4_{spc}_v6.2_monthly.nc'
    masked_ds = xr.open_dataset(spc_maskedINpath)

    ### Plotting
    # get rangeMax
    rangeMax = Abs_rangeMax_dic[spc]

    ### Accounting for scientific notation
    # Get the magnitude in scientific notation, adjust for e-
    magnitude = int(np.floor(np.log10(abs(rangeMax))))
    # Format the magnitude as a superscript
    formatted_magnitude = get_superscript(str(magnitude))
    modified_rangeMax = rangeMax*(1/(10)**magnitude)
    
    # Quick plot to check
    Timei = '2022-07-01T00:00:00.000000000'

    setmap = 'viridis'

    """ds_mappedMUSICA (regridded to ne30)"""
    PlotRegion = 'Global'

    # scalefactor = 1e-11
    # formatted_label = f"{scalefactor:.0e}" 

    Plot_ar = masked_ds.sel(time=Timei,method='nearest')['sum'].values#*scalefactor

    vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, spci_ds['sum'].molecular_weight)
    Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
    longname = masked_ds['sum'].long_name
    rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

    ### Which map
    fig = plt.figure( figsize=(8,6) ) 
    # - ne30x8 regional refinement over CONUS|
    ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
    if PlotRegion=="CONUS":
        im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
                cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
                unit=f'{formatted_magnitude}{Plot_unit}',
                state=True, lon_range=[-140,-50], lat_range=[15,60],
                  grid_line=False, grid_line_lw=0.15, twodecimal=True ) 
    elif PlotRegion=="Global":
        im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
                cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
                unit=f'{formatted_magnitude}{Plot_unit}',
                state=True, 
                  grid_line=False, grid_line_lw=0.15, twodecimal=True ) 

    plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

##### Test for Nitric Oxide Emissions (NO) | 2-D and only contains 'sum' sector

In [ ]:
spc = 'NO'
spc_fileINpath = f'{CAMS_diri}CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_{spc}_c20210423_modified.nc'

# get rangeMax
rangeMax = Abs_rangeMax_dic[spc]

### Accounting for scientific notation
# Get the magnitude in scientific notation, adjust for e-
magnitude = int(np.floor(np.log10(abs(rangeMax))))
# Format the magnitude as a superscript
formatted_magnitude = get_superscript(str(magnitude))
modified_rangeMax = rangeMax*(1/(10)**magnitude)

In [ ]:
# Open Xarray dataset
spci_ds = xr.open_dataset(spc_fileINpath)

# MUSICA lons goes from 0-360, convert it to +-180 | Need to adjust lon_right and lon_left for MUSICA!!
lat = spci_ds['lat'].values
lon = spci_ds['lon'].values
Adjustedlons = np.where(lon >= 180, lon - 360, lon)
# Create a DataArray for the adjusted lon
Adjustedlons_da = xr.DataArray(
                                Adjustedlons,  
                                dims=('ncol'),
                                coords={'ncol': spci_ds.ncol.values} 
                                )

# Add the DataArray back to the dataset
spci_ds['Adjustedlons'] = Adjustedlons_da

In [ ]:
spci_ds

In [ ]:
# Quick plot to check

Timei = '2022-07-31T00:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'CONUS'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = spci_ds.sel(time=Timei)['sum'].values#*scalefactor
Plot_unit = spci_ds.sel(time=Timei)['sum'].units#+' ('+formatted_label+')'
longname = spci_ds.sel(time=Timei)['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
# Quick plot to check

Timei = '2022-07-31T00:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'Global'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 
MonthiMean_da = spci_ds.sel(time=Timei)['sum']

Plot_ar = spci_ds.sel(time=Timei)['sum'].values#*scalefactor
Plot_unit = spci_ds.sel(time=Timei)['sum'].units#+' ('+formatted_label+')'
longname = spci_ds.sel(time=Timei)['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
spci_ds.time.values

In [ ]:
# timeslice_start = '2021-11-30T00:00:00.000000000'
# timeslice_end = '2022-11-30T00:00:00.000000000'


# seltimeMean_ds = spci_ds.sel(time=slice(timeslice_start,timeslice_end))
# latitudes = seltimeMean_ds.lat.values
# longitudes = seltimeMean_ds.Adjustedlons.values
# # seltimeMean_ds

# # Step 1: Define CONUS bounding box
# lon_lefti = -66.95   # Eastern edge
# lon_righti = -125.0  # Western edge
# lat_boti = 24.4      # Southern edge
# lat_upi = 49.5       # Northern edge

# # Step 2: Create a combined mask
# mask = np.zeros(len(lat), dtype=bool)

# for geom in gdf['geometry']:
#     for i in range(len(lat)):
#         lon = Adjustedlons[i]
#         lati = lat[i]
#         point = Point(lon, lati)

#         # Only mask if inside CONUS bounding box and also within the geometry
#         in_box = (lon >= lon_righti) & (lon <= lon_lefti) & (lati >= lat_boti) & (lati <= lat_upi)
#         if in_box and geom.contains(point):
#             mask[i] = True

# # Step 3: Convert to xarray mask
# mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": spci_ds["ncol"]})

# # Step 4: Apply the mask to remove CONUS land pixels
# masked_ds = seltimeMean_ds.where(~mask_da, drop=False)
# sum_masked = masked_ds['sum']


In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import numpy as np
import xarray as xr

# Define the time slice
timeslice_start = '2021-11-30T00:00:00.000000000'
timeslice_end = '2022-11-30T00:00:00.000000000'
seltimeMean_ds = spci_ds.sel(time=slice(timeslice_start, timeslice_end))

# Extract coordinates
lat = seltimeMean_ds['lat'].values
lon = seltimeMean_ds['Adjustedlons'].values

# Step 1: Define CONUS bounding box
lon_lefti = -66.95   # Eastern edge
lon_righti = -125.0  # Western edge
lat_boti = 24.4      # Southern edge
lat_upi = 49.5       # Northern edge

# Step 2: Use the buffered shape to mask
from shapely.geometry import Point

mask = np.zeros(len(lat), dtype=bool)
geom = gdf_buffered.iloc[0].geometry  # The buffered CONUS geometry

for i in range(len(lat)):
    point = Point(lon[i], lat[i])
    in_box = (lon[i] >= lon_righti) & (lon[i] <= lon_lefti) & (lat[i] >= lat_boti) & (lat[i] <= lat_upi)
    if in_box and geom.contains(point):
        mask[i] = True

# Step 3: Convert mask to DataArray
mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": seltimeMean_ds["ncol"]})

# Step 4: Apply the mask (keep values outside the CONUS buffered shape)
masked_ds = seltimeMean_ds.where(~mask_da, drop=False)

# Extract masked variable (optional)
sum_masked = masked_ds['sum']


In [ ]:
### Can probably save this masked array to use later 
savefilePath = f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/ne30np4_091226_pentagons_Lower48StatesCoastal50kmMaskedFalse.nc'
# Save mask_da to a NetCDF file
mask_da.to_netcdf(savefilePath)

In [ ]:
mask_da

In [ ]:
masked_ds = seltimeMean_ds.where(~mask_da, drop=False)
masked_ds

In [ ]:
### To process for all months for the given period (reduce the size of processed dataset)
Timei = '2022-07-31T00:00:00.000000000'

seltimeMean_ds = spci_ds.sel(time=Timei)
latitudes = seltimeMean_ds.lat.values
longitudes = seltimeMean_ds.Adjustedlons.values
# MonthiMean_ds

# Extract coordinates
lat = seltimeMean_ds['lat'].values
lon = seltimeMean_ds['Adjustedlons'].values

# Step 1: Define CONUS bounding box
lon_lefti = -66.95   # Eastern edge
lon_righti = -125.0  # Western edge
lat_boti = 24.4      # Southern edge
lat_upi = 49.5       # Northern edge

# Step 2: Use the buffered shape to mask
from shapely.geometry import Point

mask = np.zeros(len(lat), dtype=bool)
geom = gdf_buffered.iloc[0].geometry  # The buffered CONUS geometry

for i in range(len(lat)):
    point = Point(lon[i], lat[i])
    in_box = (lon[i] >= lon_righti) & (lon[i] <= lon_lefti) & (lat[i] >= lat_boti) & (lat[i] <= lat_upi)
    if in_box and geom.contains(point):
        mask[i] = True

# Step 3: Convert mask to DataArray
mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": seltimeMean_ds["ncol"]})

# Step 4: Apply the mask (keep values outside the CONUS buffered shape)
masked_ds = seltimeMean_ds.where(~mask_da, drop=False)

# Extract masked variable (optional)
sum_masked = masked_ds['sum']


In [ ]:
sum_masked

In [ ]:
# Quick plot to check

setmap =  'viridis' #'CMRmap_r'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'CONUS'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = masked_ds['sum'].values#*scalefactor
vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, spci_ds['sum'].molecular_weight)
Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
longname = spci_ds['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15, twodecimal=True ) 
elif PlotRegion=="Global":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            # state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15, twodecimal=True ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
# Quick plot to check

setmap = 'viridis' #'CMRmap_r'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'Global'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = masked_ds['sum'].values#*scalefactor
vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, spci_ds['sum'].molecular_weight)
Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
longname = spci_ds['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15, twodecimal=True ) 
elif PlotRegion=="Global":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            # state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15, twodecimal=True )   

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

##### Trial

In [ ]:
MonthiMean_ds = spci_ds.sel(time=Timei)
latitudes = MonthiMean_ds.lat.values
longitudes = MonthiMean_ds.Adjustedlons.values
MonthiMean_ds

In [ ]:
### First select over land
import rasterio
from rasterio.features import rasterize
# **Load High-Resolution Land Shapefile**
land_shapefile = f"{P.HOME_ROOT}/HelpfulFiles/ne_10m_land/ne_10m_land.shp"
gdf_land = gpd.read_file(land_shapefile)

# select for land pixel only
# **Define Raster Transform (Corrected)**
transform = rasterio.transform.from_bounds(
    longitudes.min(), latitudes.max(),  # Bottom-left corner
    longitudes.max(), latitudes.min(),  # Top-right corner
    len(longitudes), len(latitudes)     # Grid size
)

# **Rasterize Land Mask (1=Land, 0=Water)**
land_mask = rasterize(
    [(geom, 1) for geom in gdf_land.geometry], 
    out_shape=(len(latitudes), len(longitudes)), 
    transform=transform, 
    fill=0, 
    all_touched=True,  # Ensures land edges are included
    dtype=np.uint8
)

land_mask_xr = xr.DataArray(
    land_mask, 
    dims=["lat", "Adjustedlons"], 
    coords={"lat": MonthiMean_ds["lat"], "Adjustedlons": MonthiMean_ds["Adjustedlons"]}
)

# **Apply Mask to Dataset**
MonthiMean_da_land = MonthiMean_da.where(land_mask_xr)

In [ ]:
gdf_land

In [ ]:
### Mask values within the CONUS box (rectangular lat-lon box)
fileregion = 'CONUS_refined'

# Refined lat-lon bounding box for CONUS
lon_lefti = -66.95   # Eastern edge (near Maine)
lon_righti = -125.0  # Western edge (near California)
lat_boti = 24.4      # Southern edge (near southern Florida)
lat_upi = 49.5       # Northern edge (U.S.–Canada border)


# Define the mask for the bounding box
mask_inside = (
    (MonthiMean_ds['Adjustedlons'] >= lon_righti) &
    (MonthiMean_ds['Adjustedlons'] <= lon_lefti) &
    (MonthiMean_ds['lat'] >= lat_boti) &
    (MonthiMean_ds['lat'] <= lat_upi)
)

# Invert the mask to get data OUTSIDE the box
mask_outside = ~mask_inside

# Apply the mask to get data outside the region
outside_region_VCDds = MonthiMean_ds.where(mask_outside)


In [ ]:
outside_region_VCDds

In [ ]:
# Quick plot to check

Timei = '2022-07-31T00:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'Global'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = outside_region_VCDds['sum'].values#*scalefactor
Plot_unit = spci_ds.sel(time=Timei)['sum'].units#+' ('+formatted_label+')'
longname = spci_ds.sel(time=Timei)['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
# Quick plot to check

Timei = '2022-07-31T00:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'CONUS'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = outside_region_VCDds['sum'].values#*scalefactor
Plot_unit = spci_ds.sel(time=Timei)['sum'].units#+' ('+formatted_label+')'
longname = spci_ds.sel(time=Timei)['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
### To remove only land pixels over the CONUS (keep everythingelse)
# Remove if NOT over water & within the CONUS box 

# Step 1: Pixels outside the CONUS box
mask_outside = ~mask_inside

# Step 2: Pixels inside CONUS box but over water
mask_inside_water = mask_inside & (~land_mask)

# Step 3: Combine the two
final_mask = mask_outside | mask_inside_water

# Step 4: Apply the mask
outside_water_or_outsideCONUS_ds = MonthiMean_ds.where(final_mask)


In [ ]:
import geopandas as gpd

# Path to the shapefile (adjust if the file is inside a folder)
shapefile_path = f'{P.HOME_ROOT}/HelpfulFiles/world-administrative-boundaries/world-administrative-boundaries.shp'

# Read shapefile directly
gdf = gpd.read_file(shapefile_path)

# Preview
print(gdf.columns)
print(gdf.head())


In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely import wkt
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# Plot the GeoDataFrame
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': ccrs.PlateCarree()})
# gdf.plot(ax=ax, color='tab:blue', edgecolor='black')
gdf_buffered.plot(ax=ax, color='white', edgecolor='tab:blue')

# # Set the extent to the bounding box coordinates
# ax.set_extent([-74.5, -73.5, 40.4, 41], crs=ccrs.PlateCarree())

# Customize the plot (optional)
ax.set_title('U.S. Boundary')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
# # Create a mask for the dataset
# mask = np.zeros(len(lat), dtype=bool)

# # Iterate over the geometries and update the mask
# points_within_bounds = []
# for geom in gdf['geometry']:
#     for i in range(len(lat)):
#         point = Point(Adjustedlons[i], lat[i])
#         if geom.contains(point):
#             points_within_bounds.append((Adjustedlons[i], lat[i]))
#             mask[i] = True

# # Convert mask to DataArray
# mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": spci_ds["ncol"]})

# # Apply the mask to the dataset
# masked_ds = MonthiMean_ds.where(mask_da, drop=True)

# # Apply the mask to the 'sum' variable
# # sum_masked = spci_ds['sum'].where(~mask_da, other=np.nan)
# sum_masked = spci_ds['sum'].where(~mask_da, other=0)

In [ ]:
from shapely.geometry import Point
import numpy as np
import xarray as xr

# Step 1: Define CONUS bounding box
lon_lefti = -66.95   # Eastern edge
lon_righti = -125.0  # Western edge
lat_boti = 24.4      # Southern edge
lat_upi = 49.5       # Northern edge

# Step 2: Create a combined mask
mask = np.zeros(len(lat), dtype=bool)

for geom in gdf['geometry']:
    for i in range(len(lat)):
        lon = Adjustedlons[i]
        lati = lat[i]
        point = Point(lon, lati)

        # Only mask if inside CONUS bounding box and also within the geometry
        in_box = (lon >= lon_righti) & (lon <= lon_lefti) & (lati >= lat_boti) & (lati <= lat_upi)
        if in_box and geom.contains(point):
            mask[i] = True

# Step 3: Convert to xarray mask
mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": spci_ds["ncol"]})

# Step 4: Apply the mask to remove CONUS land pixels
masked_ds = MonthiMean_ds.where(~mask_da, drop=False) # do not drop to keep the same size

# Optionally apply it to just one variable
sum_masked = masked_ds['sum']


In [ ]:
# Step 4: Apply the mask to remove CONUS land pixels
masked_ds = MonthiMean_ds.where(~mask_da, drop=False)
sum_masked = masked_ds['sum']

In [ ]:
MonthiMean_ds

In [ ]:
masked_ds

In [ ]:
# Quick plot to check

Timei = '2022-07-31T00:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'CONUS'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = masked_ds['sum'].values#*scalefactor
Plot_unit = spci_ds.sel(time=Timei)['sum'].units#+' ('+formatted_label+')'
longname = spci_ds.sel(time=Timei)['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
# Quick plot to check

Timei = '2022-07-31T00:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'Global'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = masked_ds['sum'].values#*scalefactor
Plot_unit = spci_ds.sel(time=Timei)['sum'].units#+' ('+formatted_label+')'
longname = spci_ds.sel(time=Timei)['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

## Process for all target files

In [ ]:
Bufferopt = '80kmBuffer'
ne30maskfilePath = f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/ne30np4_091226_pentagons_CONUSlandMaskedFalse_{Bufferopt}.nc'
# Read the saved NetCDF file
mask_da_loaded = xr.open_dataarray(ne30maskfilePath)
mask_da_loaded

In [ ]:
targetspc_ls = species_names
timeslice_start = '2021-11-30T00:00:00.000000000'
timeslice_end = '2024-01-30T00:00:00.000000000'

for spc in targetspc_ls:
    spc_fileINpath = f'{CAMS_diri}CAMS-GLOB-ANT_ne30np4_{spc}_v6.2_monthly.nc'
    
    # Open Xarray dataset
    spci_ds = xr.open_dataset(spc_fileINpath)

    # MUSICA lons goes from 0-360, convert it to +-180 | Need to adjust lon_right and lon_left for MUSICA!!
    lat = spci_ds['lat'].values
    lon = spci_ds['lon'].values
    Adjustedlons = np.where(lon >= 180, lon - 360, lon)
    # Create a DataArray for the adjusted lon
    Adjustedlons_da = xr.DataArray(
                                    Adjustedlons,  
                                    dims=('ncol'),
                                    coords={'ncol': spci_ds.ncol.values} 
                                    )

    # Add the DataArray back to the dataset
    spci_ds['Adjustedlons'] = Adjustedlons_da
    
    ### Use the mask read in to remove pixels over CONUS-land, need to use zero instead of nan
    sum_masked = spci_ds['sum'].where(~mask_da_loaded, other=0)

    spci_ds['sum'] = sum_masked

    # Remove the 'Adjustedlons' variable from the dataset
    spci_ds = spci_ds.drop_vars('Adjustedlons')
    
    ### Save masked to the corresponding file
    spc_fileOUTpath = f'{CONUSlandMasked_diri}CAMS-GLOB-ANT_ne30np4_{spc}_v6.2_monthly.nc'
    spci_ds.to_netcdf(spc_fileOUTpath)
    print('Save to:',spc_fileOUTpath)
    print( '************************************************************************' )

# Estimate CONUS Emissions (numbers replaced with zero)

In [ ]:
"""Need to change"""
### Estimate emissions within NYC boundary

import os
import numpy as np
import xarray as xr
from shapely.geometry import Point

# Define dictionary to store emission sums set to zero
city_emissions = {}

# Read in each file in the format 
# for spc in updated_spcs[:2]:
for spc in updated_spcs:
    # Construct file path
    FileInPath = f'{hourlyNEI_diri}NEI2017Merged_20181MJuly_CAMSv5.1_anthro_2018_ne0CONUSne30x8_{spc}_c20231113.nc'
    
    # Check if the file exists
    if not os.path.exists(FileInPath):
        print(f"{FileInPath} missing.")
        continue

    print(f'Calculating NYC-summed July mean emissions for {spc}')
    print('************************************************************************')

    # Open the dataset
    spci_ds = xr.open_dataset(FileInPath)

    # Adjust longitudes from 0-360 to -180 to 180
    lat = spci_ds['lat'].values
    lon = spci_ds['lon'].values
    Adjustedlons = np.where(lon >= 180, lon - 360, lon)
    Adjustedlons_da = xr.DataArray(Adjustedlons, dims=('ncol'), coords={'ncol': spci_ds.ncol.values})
    spci_ds['Adjustedlons'] = Adjustedlons_da

    # Create a mask based on the boundary geometry
    mask = np.zeros(len(lat), dtype=bool)
    for geom in gdf['geometry']:
        for i in range(len(lat)):
            point = Point(Adjustedlons[i], lat[i])
            if geom.contains(point):
                mask[i] = True

    mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": spci_ds["ncol"]})

    # Initialize dictionary for this city and species if not already created
    if spc not in city_emissions:
        city_emissions[spc] = {}

    # Iterate over variables except those to ignore, apply mask, set to 0, and calculate sum
    excluded_vars = ['lon', 'lat', 'area', 'rrfac', 'date', 'Adjustedlons']
    for var_name in spci_ds.data_vars:
        if var_name not in excluded_vars:
            print(var_name)
            # Set emissions to zero within the boundary
            zeroed_emissions = spci_ds[var_name].where(~mask_da, other=0)
            
            # Calculate sum of emissions that were set to zero
            sum_Cityi_emissions = spci_ds[var_name].where(mask_da).sum().item() # use .item() to get just the number

            # Store the result in the dictionary
            if var_name not in city_emissions[spc]:
                city_emissions[spc][var_name] = sum_Cityi_emissions
                # add unit information
                city_emissions[spc]["unites"] = spci_ds["sum"].units

    print('************************************************************************')



In [ ]:
# Flatten the dictionary to create a DataFrame with separate columns for sum and units
flattened_data = {
    species: {'sum': details['sum'], 'units': details['unites']}
    for species, details in city_emissions.items()
}

# Convert to DataFrame
emissions_df = pd.DataFrame(flattened_data).T

# Format the 'sum' column in scientific notation with 2 decimal places
emissions_df['sum'] = emissions_df['sum'].apply(lambda x: f"{x:.2e}")

emissions_df

In [ ]:
# Export the DataFrame to CSV
csv_path = "./NYCsummed_JulyMean_AnthroEmis.csv"
emissions_df.to_csv(csv_path)

print("Saved to:", csv_path)

#### Draft checking calculations

In [ ]:
city_emissions

In [ ]:
zeroed_emissions.sum()

In [ ]:
var_name = 'sum'

In [ ]:
spci_ds[var_name].sum().item()

In [ ]:
spci_ds[var_name].where(mask_da).sum()

In [ ]:
zeroed_emissions.sum()+spci_ds[var_name].where(mask_da).sum()

In [ ]:
['lon','lat','area','rrfac','date','Adjustedlons']

# Analyze Output Simulations

# Remove CMIP6-Aircraft emissions over CONUS Box-land

In [ ]:
# ### Locate aircraft emissions fires
# aircraft_emisfilepaths = [
# '{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_SO2_aircraft_verticto2025use2011to2015_.9x1.25_c20170608.nc',
# '{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_num_bc_a4_aircraft_verticto2025use2011to2015_.9x1.25_c20170608.nc',
# '{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_bc_a4_aircraft_verticto2025use2011to2015_.9x1.25_c20170608.nc',
# '{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_NO2_aircraft_verticto2025use2011to2015_.9x1.25_c20170608.nc',
# ]

In [ ]:
### Locate aircraft emissions fires
aircraft_emisfilepaths = [
f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/approx2022/emissions-cmip6_SO2_aircraft_vertical_to2022use2015_0.9x1.25_c20170608.nc',
f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/approx2022/emissions-cmip6_num_bc_a4_aircraft_vertical_to2022use2015_0.9x1.25_c20170608.nc',
f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/approx2022/emissions-cmip6_bc_a4_aircraft_vertical_to2022use2015_0.9x1.25_c20170608.nc',
f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/approx2022/emissions-cmip6_NO2_aircraft_vertical_to2022use2015_0.9x1.25_c20170608.nc',
]

In [ ]:
# Read one file in to create a mask for f09 (.9x1.25) resolution
spc_fileINpath = f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/approx2022/emissions-cmip6_NO2_aircraft_vertical_to2022use2015_0.9x1.25_c20170608.nc'
# Open Xarray dataset
spci_ds = xr.open_dataset(spc_fileINpath)
spci_ds

In [ ]:
from shapely.geometry import Point
import numpy as np
import xarray as xr

Bufferopt = '80kmBuffer'

# Adjust longitudes from 0–360 to -180–180
Adjustedlons = np.where(spci_ds['lon'] >= 180, spci_ds['lon'] - 360, spci_ds['lon'])
spci_ds['Adjustedlons'] = ('lon', Adjustedlons)

# Use a single time slice
Timei = '2022-07-31T00:00:00.000000000'
MonthiMean_ds = spci_ds.sel(time=Timei, method="nearest")

latitudes = MonthiMean_ds['lat'].values
longitudes = MonthiMean_ds['Adjustedlons'].values

# Define CONUS bounding box
lon_lefti = -66.95
lon_righti = -125.0
lat_boti = 24.4
lat_upi = 49.5

# Create a 2D mask array (lat × lon)
mask = np.zeros((len(latitudes), len(longitudes)), dtype=bool)

# Loop over lat-lon grid
for i, lat_val in enumerate(latitudes):
    for j, lon_val in enumerate(longitudes):
        point = Point(lon_val, lat_val)
        in_box = (lon_righti <= lon_val <= lon_lefti) and (lat_boti <= lat_val <= lat_upi)
        if in_box:
            for geom in gdf_buffered['geometry']:
                if geom.contains(point):
                    mask[i, j] = True
                    break  # No need to check other geometries if already contained

# Step 3: Convert to xarray DataArray
mask_da = xr.DataArray(
    mask,
    dims=["lat", "lon"],
    coords={"lat": MonthiMean_ds["lat"], "lon": MonthiMean_ds["lon"]},
    name="CONUS_mask"
)

# Step 4: Apply the mask to remove CONUS land pixels
masked_ds = MonthiMean_ds.where(~mask_da, drop=False)

# Optional: Save the mask to a NetCDF file
maskfilePath = f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/f09_CONUSlandMaskedFalse_{Bufferopt}.nc'
mask_da.to_netcdf(maskfilePath)
print('Saved mask to:', maskfilePath)


In [ ]:
MonthiMean_ds

In [ ]:
Bufferopt = '80kmBuffer'
f09maskfilePath = f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/f09_CONUSlandMaskedFalse_{Bufferopt}.nc'
# Read the saved NetCDF file
mask_da_loaded = xr.open_dataarray(f09maskfilePath)
mask_da_loaded

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig = plt.figure(figsize=(12, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
mask_da_loaded.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='gray_r', add_colorbar=False)

ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.set_title("CONUS Coastal Mask", fontsize=16)
ax.set_extent([-130, -60, 20, 55], crs=ccrs.PlateCarree())  # Zoom to US
plt.grid(True)
plt.show()


In [ ]:
### Read in original emissions data
Original_diri = f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/approx2022/'

CONUSlandMasked_diri = f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4_CONUSlandMasked_c20250619/'

In [ ]:
### Locate aircraft emissions fires
aircraft_emisfiles = [
# 'emissions-cmip6_SO2_aircraft_vertical_to2022use2015_0.9x1.25_c20170608.nc',
'emissions-cmip6_num_bc_a4_aircraft_vertical_to2022use2015_0.9x1.25_c20170608.nc',
'emissions-cmip6_bc_a4_aircraft_vertical_to2022use2015_0.9x1.25_c20170608.nc',
# 'emissions-cmip6_NO2_aircraft_vertical_to2022use2015_0.9x1.25_c20170608.nc',
]

In [ ]:
targetspc_ls = species_names
timeslice_start = '2021-11-30T00:00:00.000000000'
timeslice_end = '2024-01-30T00:00:00.000000000'

for spc_fileIN in aircraft_emisfiles:   
    spc_fileINpath = f'{Original_diri}{spc_fileIN}'
    # Open dataset
    spci_ds = xr.open_dataset(spc_fileINpath)

    # Subset by time if needed
    if "time" in spci_ds.dims:
        spci_ds = spci_ds.sel(time=slice(timeslice_start, timeslice_end))

    # Apply CONUS land mask (mask_da_loaded is True over CONUS land)
    # Replace masked values with zero instead of NaN
    if "emiss_aircraft" in spci_ds:
        sum_masked = spci_ds["emiss_aircraft"].where(~mask_da_loaded, other=0)
        spci_ds["emiss_aircraft"] = sum_masked
    elif "num_bc_a4_aircraft" in spci_ds:
        sum_masked = spci_ds["num_bc_a4_aircraft"].where(~mask_da_loaded, other=0)
        spci_ds["num_bc_a4_aircraft"] = sum_masked
    else:
        raise ValueError(f"'Check for the valid variable name in {spc_fileINpath}")

    # Generate species name from filename if not explicitly passed
    spc = None
    for candidate in targetspc_ls:
        if candidate in spc_fileINpath:
            spc = candidate
            break
    if spc is None:
        raise ValueError(f"Could not match species name in: {spc_fileINpath}")

    # Save to output
    spc_fileOUTpath = f"{CONUSlandMasked_diri}{spc_fileIN}"
    spci_ds.to_netcdf(spc_fileOUTpath)
    
    print("Saved to:", spc_fileOUTpath)
    print("**********************************************************")


In [ ]:
spci_ds

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Select 2D slice from DataArray, not Dataset
plotda = spci_ds['emiss_aircraft'].isel(altitude=0,time=1)

fig = plt.figure(figsize=(12, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
plotda.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='viridis', cbar_kwargs={"label": "emiss_aircraft"})
ax.coastlines()
ax.add_feature(cfeature.BORDERS)
ax.set_title("Masked Aircraft Emissions (alt=0)")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Select 2D slice from DataArray
plotda = spci_ds['emiss_aircraft'].isel(altitude=0, time=1)

fig = plt.figure(figsize=(12, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

# Set extent to CONUS: [west, east, south, north]
ax.set_extent([-130, -65, 24, 50], crs=ccrs.PlateCarree())

# Plot with better color normalization (log or custom range for contrast)
plotda.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap='viridis',
    vmin=plotda.where(plotda > 0).quantile(0.05),  # optional: cut off very low values
    vmax=plotda.quantile(0.95),                    # optional: cut off high outliers
    cbar_kwargs={"label": "Aircraft Emissions (units?)"}
)

# Add map features
ax.coastlines()
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.STATES, linewidth=0.5)
ax.set_title("Masked Aircraft Emissions over the US (alt=0)")

plt.tight_layout()
plt.show()


# Remove Biomass Burning emissions over CONUS Box-land